In [14]:
# check point
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver  # this is in-built checkpointer that saves data after every execution in thread
from langchain_core.runnables import RunnableConfig
from typing import Annotated, TypedDict 
from operator import add 

class State(TypedDict):
    foo: str 
    bar: Annotated[list[str], add]

def node_e(state: State):
    return {'foo': 'a', 'bar': ['a']}

def node_b(state: State):
    return {'foo': 'b', 'bar': ['b']}

workflow = StateGraph(State)

workflow.add_node(node_e)
workflow.add_node(node_b)
workflow.add_edge(START, 'node_e')
workflow.add_edge('node_e', 'node_b')
workflow.add_edge('node_b', END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {'configurable': {'thread_id': '1'}}

In [17]:
graph.invoke({'foo': ''}, config)

{'foo': 'b', 'bar': ['a', 'b', 'a', 'b', 'a', 'b']}

In [3]:
graph.get_state(config)

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0957a4-b66f-6c53-8002-58cd57d933e0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-09-19T17:01:33.207431+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0957a4-b66b-697d-8001-09f614c3a2b2'}}, tasks=(), interrupts=())

In [9]:
list(graph.get_state_history(config))

[StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0957a4-b66f-6c53-8002-58cd57d933e0'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-09-19T17:01:33.207431+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0957a4-b66b-697d-8001-09f614c3a2b2'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0957a4-b66b-697d-8001-09f614c3a2b2'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-09-19T17:01:33.205719+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0957a4-b666-6c53-8000-a184f13b3b2a'}}, tasks=(PregelTask(id='fdbd5d8f-f688-ed02-4dab-3f3aad145ec8', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts

In [10]:
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1f0957a4-b66b-697d-8001-09f614c3a2b2"}}
graph.invoke(None, config=config)

{'foo': 'b', 'bar': ['a', 'b']}

In [18]:
in_memory_store = InMemorySaver()
user_id = 'i1'
namespace_for_memory = (user_id, 'memories')

In [20]:
import uuid

#### embeddding model

In [23]:
from dotenv import load_dotenv

load_dotenv()

True

In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector = embeddings.embed_query("hello world")
len(vector)

/home/chetan/anaconda3/envs/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


3072

#### Connecting to pinecone

In [6]:
from pinecone import Pinecone, ServerlessSpec

index_name = "med-ai-test"
import os
# os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
# PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
# os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
pc = Pinecone()


In [7]:
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

# index =/\ pc.Index

In [12]:
import os
import uuid 
# from datetime import datatime
import datetime
from pymongo import MongoClient 
from dotenv import load_dotenv 

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END

In [11]:
load_dotenv()

True

In [23]:
MONGO_URI = os.getenv('MONGO_URI')
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
PINECONE_INDEX = os.getenv('PINECONE_INDEX')
# pinecone index name should also be in .env
# but for now we are skipping that part

mongo = MongoClient(MONGO_URI)
db = mongo.get_database('test')
messages_col = db.messages


llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')

In [26]:
vectorStore = PineconeVectorStore.from_existing_index(
    index_name='med-ai-test',
    embedding=embeddings
)

In [ ]:
def store_messages(user_id: str, conversation_id: str, role: str, content: str, metadata=None):
    doc = {
        'usedId': user_id,
        'conversationId': conversation_id,
        'role': role,
        'content': content,
        'createdAt': datetime.utcnow(),
        'metadata': metadata or {}
    }

    res = messages_col.insert_one(doc)
    mongo_id = str(res.inserted_id)

    lc_doc = Document(
        page_content=content,
        metadata={
            'userId': user_id,
            'conversationId': conversation_id,
            'role': role,
            'mongoIda': mongo_id,
            'createdAt': doc['createdAt'].isoformat()
        }
    )
    vectorStore.add_documents([lc_doc])
    return {'mongoId': mongo_id}

In [1]:
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=3)

In [3]:
res = search.invoke("prime minister of india")

In [4]:
res

{'query': 'prime minister of india',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.pmindia.gov.in/en/pms-profile/',
   'title': 'Know the PM | Prime Minister of India',
   'content': 'Shri Narendra Modi was sworn-in as India’s Prime Minister for the third time on 9th June 2024, following another decisive victory in the 2024 Parliamentary elections. The first ever Prime Minister to be born after Independence, Shri Modi has previously served as the Prime Minister of India from 2014 to 2019, and from 2019 to 2024. Leading international agencies have noted that under the leadership of PM Narendra Modi, India has been eliminating poverty at record pace. Shri Modi believes that no Indian should be homeless and to realise this vision, over 4.2 crore houses were sanctionedunder the PM Awas Yojana between 2014 and 2024. PM Modi launched the ‘Make in India’ initiative to turn India into an international manufacturing powerhouse.',
   'score': 0.8

In [17]:
l = ""
for i in res['results']:
        l = l + i['content']

In [20]:
l[:500]

'Shri Narendra Modi was sworn-in as India’s Prime Minister for the third time on 9th June 2024, following another decisive victory in the 2024 Parliamentary elections. The first ever Prime Minister to be born after Independence, Shri Modi has previously served as the Prime Minister of India from 2014 to 2019, and from 2019 to 2024. Leading international agencies have noted that under the leadership of PM Narendra Modi, India has been eliminating poverty at record pace. Shri Modi believes that no '

In [21]:
from langchain_core.tools import tool

In [ ]:
@tool 
def search(query: str) -> str:
    """Takes a query and perform web search"""
    res = search.invoke(query)

    l = ""
    for i in res['results']:
        l = l + i['content']

    l = l[:500]

    return l

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq


# from tools.tools import calculator, search
# from schema.schema import ClassifierModelSchema



from dotenv import load_dotenv

load_dotenv()

# tools = [calculator, search]
# tool_node = ToolNode(tools)

# llm = HuggingFaceEndpoint(
#     repo_id='openai/gpt-oss-120b'
# )
# refiner_model = ChatHuggingFace(
#     llm=llm
# )

summarizer_llm = HuggingFaceEndpoint(
    # repo_id='openai/gpt-oss-120b'
    repo_id='Qwen/Qwen3-Next-80B-A3B-Instruct'
)
summary_model = ChatHuggingFace(llm=summarizer_llm)

In [5]:
summary_model.invoke("hello this is the intro")

AIMessage(content="Hello! 👋  \nThanks for saying hello — I'm here and ready to help with whatever you need. Whether it's answering questions, brainstorming ideas, or just chatting, I've got your back! 😊  \nWhat would you like to do next?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 13, 'total_tokens': 66}, 'model_name': 'Qwen/Qwen3-Next-80B-A3B-Instruct', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='run--5e8603b0-16d5-41c2-a71f-8fccea7f0822-0', usage_metadata={'input_tokens': 13, 'output_tokens': 53, 'total_tokens': 66})

In [2]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

In [3]:
msg = [HumanMessage(content='find nearest hospital, my location is nit kurukshetra', additional_kwargs={}, response_metadata={}, id='233b943c-9cd8-4673-af70-c792cb46b8be'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants nearest hospital, location is "nit kurukshetra". We need to provide location-specific info. We can use search tool to find nearest hospitals to NIT Kurukshetra. Use search query.', 'tool_calls': [{'index': 0, 'id': 'fc_ce698d08-039d-4884-a0d3-2901efdf56e0', 'function': {'arguments': '{"query":"nearest hospital to NIT Kurukshetra"}', 'name': 'search'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_91af62a853', 'service_tier': 'on_demand'}, id='run--ef91a060-a2f0-4335-86ea-647722d97289', tool_calls=[{'name': 'search', 'args': {'query': 'nearest hospital to NIT Kurukshetra'}, 'id': 'fc_ce698d08-039d-4884-a0d3-2901efdf56e0', 'type': 'tool_call'}], usage_metadata={'input_tokens': 343, 'output_tokens': 79, 'total_tokens': 422}), ToolMessage(content='Pawan Surgical Hospital, Near Old. Bus Stand, Red Road, KKR. Surgeon ... Maharaja Agarsen Hospital, Near Sai. Mandir, Ratgal Road, Opp. Maharaja. Pratap1. Dr. Rishi Pal Gupta, · 2. Cygnus Hospital. Opp. · 3. Kurukshetra Nursing Home, Pipli. Road, Kurukshetra · 4. Saraswti Mission Hospital, Opp Neel. Kanth YatriPipli Road, Near New Bus stand, Kurukshetra, 9416039397, Click here to Go. 60 ... Santosh Multi Specialty Hospital, 3F-139, NIT, Near Police Post, Faridabad', name='search', id='7bb1d83b-c278-457e-bc13-d598d7c4fcaa', tool_call_id='fc_ce698d08-039d-4884-a0d3-2901efdf56e0')]

In [5]:
for i in msg:
    print(i.content)

find nearest hospital, my location is nit kurukshetra

Pawan Surgical Hospital, Near Old. Bus Stand, Red Road, KKR. Surgeon ... Maharaja Agarsen Hospital, Near Sai. Mandir, Ratgal Road, Opp. Maharaja. Pratap1. Dr. Rishi Pal Gupta, · 2. Cygnus Hospital. Opp. · 3. Kurukshetra Nursing Home, Pipli. Road, Kurukshetra · 4. Saraswti Mission Hospital, Opp Neel. Kanth YatriPipli Road, Near New Bus stand, Kurukshetra, 9416039397, Click here to Go. 60 ... Santosh Multi Specialty Hospital, 3F-139, NIT, Near Police Post, Faridabad
